
# **import the libraries**

In [2]:
import pandas as pd
devto_file_path = '../data/raw/devto_9-9-26_raw_1k_records.json'

# **Function for easy profiling**


In [3]:


def profile_data(df, source_name="Dataset"):
    """
    Generates a profiling summary table and prints dataset dimensions.
    Safely handles unhashable types (like lists/dicts) in unique counts.
    """
    print(f"--- Profiling Report: {source_name} ---")
    print(f"Row Count: {df.shape[0]}")
    print(f"Column Count: {df.shape[1]}\n")

    # Safely count uniques by treating lists/dicts as strings
    safe_unique_counts = [df[col].astype(str).nunique() for col in df.columns]

    # Generate the exact table required by the Task 2 deliverables
    profiling_summary = pd.DataFrame({
        'Data Type': df.dtypes,
        'Null Count': df.isnull().sum(),
        'Percent Null': (df.isnull().sum() / len(df)) * 100,
        'Unique Count': safe_unique_counts,
        'Sample Value': df.iloc[0] if not df.empty else None
    })

    return profiling_summary

# **Read JSON file devto_9-9-26_raw_1k_records.json using pandas**

In [14]:
import json
list_column_name = "tags"
# 1. Read the raw JSON file
with open(devto_file_path, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

# 2. Normalize and flatten the data
df = pd.json_normalize(raw_data)

# 3. Convert the list elements into a new DataFrame side-by-side
expanded_cols = pd.DataFrame(df[list_column_name].tolist())

# 4. Generate dynamic column names (e.g., hobbies_1, hobbies_2)
expanded_cols.columns = [f"{list_column_name}_{i+1}" for i in range(expanded_cols.shape[1])]

# 5. Drop the original list column and join the new columns back
df_devto = df.drop(columns=[list_column_name]).join(expanded_cols)

# View the final result
df_devto.head()




,article_id,title,author,published_date,url,reactions_count,comments_count,summary,tags_1,tags_2,tags_3,tags_4
0,4603458,Most 'AI Agents' Are Just If-Statements in a T...,James Anderson,2026-09-08T06:12:03Z,https://dev.to/james_anderson_h/most-ai-agents...,44,38,"I built an agent last year, and I was proud of...",ai,softwareengineering,webdev,agents
1,4608508,Why I built a zero-telemetry Chrome extension ...,IB2M_Official,2026-09-08T17:46:26Z,https://dev.to/ib2mofficial/why-i-built-a-zero...,3,2,Every developer today relies on LLMs. Whether ...,webdev,ai,productivity,extensions
2,4614948,I Tested generateStaticParams in Next.js Cache...,Shubhra Pokhariya,2026-09-09T12:22:33Z,https://dev.to/shubhradev/i-tested-generatesta...,7,0,The last post here got picked apart in the com...,webdev,javascript,react,nextjs
3,4606981,Rescue Me: ember-website,Isaac Lee,2026-09-08T13:18:04Z,https://dev.to/ijlee2/rescue-me-ember-website-...,0,0,I modernized ember-website over the past week....,ember,opensource,platform,webdev
4,4608200,What Happens When You Type a URL: Answer It Wi...,DevOps Daily,2026-09-08T16:43:00Z,https://dev.to/devopsdaily/what-happens-when-y...,18,5,"""What happens when you type a URL and press en...",networking,dns,tcp,webdev


Profiling for devto_9-9-26_raw_1k_records.json

In [15]:
profile_data(df_devto, source_name= "Dev to" )

--- Profiling Report: Dev to ---
Row Count: 1000
Column Count: 12



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
article_id,int64,0,0.0,1000,4603458
title,str,0,0.0,830,Most 'AI Agents' Are Just If-Statements in a T...
author,str,0,0.0,673,James Anderson
published_date,str,0,0.0,995,2026-09-08T06:12:03Z
url,str,0,0.0,1000,https://dev.to/james_anderson_h/most-ai-agents...
reactions_count,int64,0,0.0,17,44
comments_count,int64,0,0.0,7,38
summary,str,0,0.0,887,"I built an agent last year, and I was proud of..."
tags_1,str,0,0.0,135,ai
tags_2,str,2,0.2,136,softwareengineering


checking duplicate titles

In [16]:
# Count how many times each title appears
title_counts = df_devto['title'].value_counts()

# Filter to show only the titles that appear more than once
duplicated_titles = title_counts[title_counts > 1]

print("Titles with duplicates:")
display(duplicated_titles.head(10))

Titles with duplicates:


title
USDC Escrow for AI Agents: How Trustless Freelancing Actually Works              49
x402 Explained: HTTP-Native Micropayments for AI Agents (With Real Code)         39
From Prompt to Paycheck: Wiring an LLM Chain Into Real Gig Platforms             33
How I Built an Autonomous AI Agent That Earns USDC While I Sleep                 29
The Complete Guide to Agent-to-Agent Marketplaces in 2026                        19
I built 59 free browser-based dev tools in vanilla JS — here's what I learned     6
Docker Alternatives for Local Postgres Dev: A 2026 Comparison                     2
Name: count, dtype: int64

In [17]:
dup_title = df_devto[df_devto['title'] == "USDC Escrow for AI Agents: How Trustless Freelancing Actually Works"]
pd.set_option('display.max_colwidth', None)
dup_title['url'].head()

263    https://dev.to/nikhilranka23/usdc-escrow-for-ai-agents-how-trustless-freelancing-actually-works-3eoc
275    https://dev.to/nikhilranka23/usdc-escrow-for-ai-agents-how-trustless-freelancing-actually-works-4d1c
354     https://dev.to/nikhilranka23/usdc-escrow-for-ai-agents-how-trustless-freelancing-actually-works-1pi
363    https://dev.to/nikhilranka23/usdc-escrow-for-ai-agents-how-trustless-freelancing-actually-works-3e32
371     https://dev.to/nikhilranka23/usdc-escrow-for-ai-agents-how-trustless-freelancing-actually-works-6jm
Name: url, dtype: str

Drop duplicates based on title and author

In [ ]:
# 1. Sort by date so the newest version of the article is first
df_devto = df_devto.sort_values('published_date', ascending=False)

# 2. Drop duplicates looking ONLY at the title and author columns, keeping the first (newest) row
df_devto = df_devto.drop_duplicates(subset=['title', 'author'], keep='first')

# 3. Check how many rows are left
print(f"Rows remaining after removing title spam: {len(df_medium)}")

Rows remaining after removing title spam: 831


checking profile again

In [18]:
profile_data(df_devto, source_name= "devto")

--- Profiling Report: devto ---
Row Count: 1000
Column Count: 12



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
article_id,int64,0,0.0,1000,4603458
title,str,0,0.0,830,Most 'AI Agents' Are Just If-Statements in a Trench Coat
author,str,0,0.0,673,James Anderson
published_date,str,0,0.0,995,2026-09-08T06:12:03Z
url,str,0,0.0,1000,https://dev.to/james_anderson_h/most-ai-agents-are-just-if-statements-in-a-trench-coat-3960
reactions_count,int64,0,0.0,17,44
comments_count,int64,0,0.0,7,38
summary,str,0,0.0,887,"I built an agent last year, and I was proud of it. It had a planner. It had tools. It had a..."
tags_1,str,0,0.0,135,ai
tags_2,str,2,0.2,136,softwareengineering


investigate duplicate titles

In [ ]:
# Count how many times each summary appears
summary_counts = df_devto['summary'].value_counts()

# Filter to show only the titles that appear more than once
duplicated_summaries = summary_counts[summary_counts > 1]

print("summaries with duplicates:")
pd.set_option('display.max_colwidth', None)
duplicated_summaries.head(10)

summaries with duplicates:


summary
An honest take on where AI agents, LLMs, and production systems actually are right now -- from someone deep in the space.    4
![Architecture...                                                                                                            4
GPT-6 Astra, Claude Fable, Gemini 3.8: A Busy Week for New LLM Releases   Weekly digest from...                              2
Liquid syntax error: Variable '{{ uri: {% raw %}' was not properly terminated with regexp: /\}\}/                            2
Name: count, dtype: int64

checking duplicate summaries

In [ ]:
df_devto[df_devto["summary"] == "An honest take on where AI agents, LLMs, and production systems actually are right now -- from someone deep in the space."]

,article_id,title,author,published_date,url,tags,reactions_count,comments_count,summary
102,4610795,What Nobody Tells You About Deploying LLMs at Scale,AI Bug Slayer 🐞,2026-09-09T03:34:44Z,https://dev.to/aibughunter/what-nobody-tells-you-about-deploying-llms-at-scale-1f1,"[aiagents, ai, llm, webdev]",0,0,"An honest take on where AI agents, LLMs, and production systems actually are right now -- from someone deep in the space."
103,4610784,How a Single AI Agent Replaced a 5-Person Data Team at a Fintech Startup,AI Bug Slayer 🐞,2026-09-09T03:32:37Z,https://dev.to/aibughunter/how-a-single-ai-agent-replaced-a-5-person-data-team-at-a-fintech-startup-4pm0,"[aiagents, ai, llm, webdev]",0,0,"An honest take on where AI agents, LLMs, and production systems actually are right now -- from someone deep in the space."
759,4591566,The LLM Nobody Talks About That Keeps Showing Up in Production Stacks,AI Bug Slayer 🐞,2026-09-07T03:33:54Z,https://dev.to/aibughunter/the-llm-nobody-talks-about-that-keeps-showing-up-in-production-stacks-ghl,"[aiagents, ai, llm, webdev]",0,0,"An honest take on where AI agents, LLMs, and production systems actually are right now -- from someone deep in the space."
765,4591558,"Two Years From Now, This Will Be the Only Skill That Matters in AI",AI Bug Slayer 🐞,2026-09-07T03:32:22Z,https://dev.to/aibughunter/two-years-from-now-this-will-be-the-only-skill-that-matters-in-ai-2mk3,"[aiagents, ai, llm, webdev]",0,0,"An honest take on where AI agents, LLMs, and production systems actually are right now -- from someone deep in the space."


In [ ]:
df_devto[df_devto["summary"] == "Liquid syntax error: Variable '{{ uri: {% raw %}' was not properly terminated with regexp: /\}\}/"]

<>:1: SyntaxWarning: invalid escape sequence '\}'
<>:1: SyntaxWarning: invalid escape sequence '\}'
C:\Users\Azam\AppData\Local\Temp\ipykernel_76992\552985473.py:1: SyntaxWarning: invalid escape sequence '\}'
  df_devto[df_devto["summary"] == "Liquid syntax error: Variable '{{ uri: {% raw %}' was not properly terminated with regexp: /\}\}/"]


,article_id,title,author,published_date,url,tags,reactions_count,comments_count,summary
87,4611831,I Profiled 12 React Native Screens. The Same Three Mistakes Were in All of Them. published: true,Arpit Mishra,2026-09-09T06:13:09Z,https://dev.to/arpit_mishra1/i-profiled-12-react-native-screens-the-same-three-mistakes-were-in-all-of-thempublished-true-o49,"[ai, programming, webdev, security]",0,0,Liquid syntax error: Variable '{{ uri: {% raw %}' was not properly terminated with regexp: /\}\}/
86,4611813,I Profiled 12 React Native Screens. The Same Three Mistakes Were in All of Them.,Arpit Mishra,2026-09-09T06:11:33Z,https://dev.to/arpit_mishra1/i-profiled-12-react-native-screens-the-same-three-mistakes-were-in-all-of-them-5gd,"[ai, webdev, productivity, programming]",0,0,Liquid syntax error: Variable '{{ uri: {% raw %}' was not properly terminated with regexp: /\}\}/


removing duplicate titles with autho

In [ ]:
# 1. Sort by date so the newest version of the article is first
df_devto = df_devto.sort_values('published_date', ascending=False)

# 2. Drop duplicates looking ONLY at the summary columns, keeping the first (newest) row
df_devto = df_devto.drop_duplicates(subset=['summary'], keep='first')

# 3. Check how many rows are left
print(f"Rows remaining after removing summary title spam: {len(df_devto)}")

Rows remaining after removing summary title spam: 823


profile after changes

In [ ]:
profile_data(df_devto, source_name= "devto")

--- Profiling Report: devto ---
Row Count: 823
Column Count: 9



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
article_id,int64,0,0.0,823,4615546
title,object,0,0.0,822,"My English page shipped with Japanese error messages, because the logic owned the strings"
author,object,0,0.0,673,takahiro hashito
published_date,object,0,0.0,818,2026-09-09T13:32:14Z
url,object,0,0.0,823,https://dev.to/takahiro_hashito_a1f3f0dc/my-english-page-shipped-with-japanese-error-messages-because-the-logic-owned-the-strings-40c4
tags,object,0,0.0,724,"[javascript, i18n, webdev, refactoring]"
reactions_count,int64,0,0.0,17,0
comments_count,int64,0,0.0,7,0
summary,object,0,0.0,823,Background I ship small single-file browser tools. Every tool has a Japanese page and an...


Second source: KDnuggets

In [ ]:
KDnuggets_file_path = '../data/raw/KDnuggets_10-9-26_150_records.json'
df_KDnuggets = pd.read_json(KDnuggets_file_path)
df_KDnuggets.head()

,title,author,published_date,url,tags,summary
0,7 Steps to Become a Forward Deployed Engineer in 2026,Kanwal Mehreen,"Thu, 10 Sep 2026 14:00:10 +0000",https://www.kdnuggets.com/7-steps-to-become-a-forward-deployed-engineer-in-2026,NaN,FDEs are becoming some of the most in-demand engineers in AI. Here’s the 7-step roadmap to becoming one in 2026.
1,5 Useful Python Scripts to Automate CSV Processing,Bala Priya C,"Thu, 10 Sep 2026 12:00:50 +0000",https://www.kdnuggets.com/5-useful-python-scripts-to-automate-csv-processing,NaN,"Automate common CSV tasks with these 5 Python scripts for cleaning, validating, transforming, and processing CSV files using the standard library."
2,Build an AI Data Analyst That Thinks Like a Senior Analyst,Nate Rosidi,"Wed, 09 Sep 2026 14:00:50 +0000",https://www.kdnuggets.com/build-an-ai-data-analyst-that-thinks-like-a-senior-analyst,NaN,A six-stage pipeline that checks its numbers before calling anything an answer.
3,7 Approaches to Efficient LLM Training on Limited Hardware,Vinod Chugani,"Wed, 09 Sep 2026 12:00:50 +0000",https://www.kdnuggets.com/7-approaches-to-efficient-llm-training-on-limited-hardware,NaN,Learn seven engineering techniques to train large language models on consumer GPUs without running out of memory.
4,From RAG to Agentic AI: Building the Next Generation of Intelligent Enterprise Systems,Mona Sachdev,"Tue, 08 Sep 2026 16:00:12 +0000",https://www.kdnuggets.com/from-rag-to-agentic-ai-building-the-next-generation-of-intelligent-enterprise-systems,NaN,"Over the past several years, I have worked through three successive generations of intelligent retrieval systems, each solving problems the previous generation could not. Here is what I have learned."


Profile for KDnuggets

In [ ]:
profile_data(df_KDnuggets, source_name= "KDnuggets")

--- Profiling Report: KDnuggets ---
Row Count: 150
Column Count: 6



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
title,object,0,0.0,10,7 Steps to Become a Forward Deployed Engineer in 2026
author,object,0,0.0,6,Kanwal Mehreen
published_date,object,0,0.0,10,"Thu, 10 Sep 2026 14:00:10 +0000"
url,object,0,0.0,10,https://www.kdnuggets.com/7-steps-to-become-a-forward-deployed-engineer-in-2026
tags,float64,150,100.0,1,NaN
summary,object,0,0.0,10,FDEs are becoming some of the most in-demand engineers in AI. Here’s the 7-step roadmap to becoming one in 2026.


third source medium

In [ ]:
medium_file_path = '../data/raw/medium_10-9-26_raw_268_records.json'
df_medium = pd.read_json(medium_file_path)
df_medium.head()

,title,author,published_date,url,tags,summary
0,CREATING CONSISTENT POSTS WITH QUOCIA,Candy Chan,"Thu, 10 Sep 2026 12:31:51 GMT",https://medium.com/@candy_cyy/creating-consistent-posts-with-quocia-aa316f8a406f?source=rss------artificial_intelligence-5,"[marketing, artificial-intelligence, quocia, ai, social-media]","<div class=""medium-feed-item""><p class=""medium-feed-image""><a href=""https://medium.com/@candy_cyy/creating-consistent-posts-with-quocia-aa316f8a406f?source=rss------artificial_intelligence-5""><img src=""https://cdn-images-1.medium.com/max/1408/1*Mykxoyetl0mRaZF7Ovi1dg.jpeg"" width=""1408"" /></a></p><p class=""medium-feed-snippet"">Consistency is the single most decisive factor separating thriving digital brands from those that fade into irrelevance.</p><p class=""medium-feed-link""><a href=""https://medium.com/@candy_cyy/creating-consistent-posts-with-quocia-aa316f8a406f?source=rss------artificial_intelligence-5"">Continue reading on Medium »</a></p></div>"
1,WHITE PAPER: The Great Deception,Christopher Frederick,"Thu, 10 Sep 2026 12:31:41 GMT",https://medium.com/@kitsagr8m8/white-paper-the-great-deception-f543c20e8da6?source=rss------artificial_intelligence-5,"[stock-market, quantitative-finance, macroeconomics, artificial-intelligence, geopolitical]","<div class=""medium-feed-item""><p class=""medium-feed-snippet"">The Convergence of the AI Liquidity Bubble, Resource Constraints, and the Geopolitical Flashpoint</p><p class=""medium-feed-link""><a href=""https://medium.com/@kitsagr8m8/white-paper-the-great-deception-f543c20e8da6?source=rss------artificial_intelligence-5"">Continue reading on Medium »</a></p></div>"
2,O Empresário do Futuro Parece Mais um Maestro do que um Gestor,Francisco Santos,"Thu, 10 Sep 2026 12:31:37 GMT",https://medium.com/@franciscosantoscientista/o-empres%C3%A1rio-do-futuro-parece-mais-um-maestro-do-que-um-gestor-724ad1ab0d2b?source=rss------artificial_intelligence-5,"[innovation, business-development, business, artificial-intelligence, business-strategy]","<div class=""medium-feed-item""><p class=""medium-feed-image""><a href=""https://medium.com/@franciscosantoscientista/o-empres%C3%A1rio-do-futuro-parece-mais-um-maestro-do-que-um-gestor-724ad1ab0d2b?source=rss------artificial_intelligence-5""><img src=""https://cdn-images-1.medium.com/max/1672/1*jFqiUnqML18CblQumyulXA.png"" width=""1672"" /></a></p><p class=""medium-feed-snippet"">Durante d&#xe9;cadas, ser empres&#xe1;rio significava ser a fonte central de todas as respostas. Saber vender, negociar, apagar inc&#xea;ndios, controlar&#x2026;</p><p class=""medium-feed-link""><a href=""https://medium.com/@franciscosantoscientista/o-empres%C3%A1rio-do-futuro-parece-mais-um-maestro-do-que-um-gestor-724ad1ab0d2b?source=rss------artificial_intelligence-5"">Continue reading on Medium »</a></p></div>"
3,AI Wants Our Work. Ask Us First.,John Adam Ledford,"Thu, 10 Sep 2026 12:31:01 GMT",https://johnadamledford.medium.com/ai-wants-our-work-ask-us-first-b39c145661dc?source=rss------artificial_intelligence-5,"[creator-economy, music, copyright, artificial-intelligence, writing]","<div class=""medium-feed-item""><p class=""medium-feed-image""><a href=""https://johnadamledford.medium.com/ai-wants-our-work-ask-us-first-b39c145661dc?source=rss------artificial_intelligence-5""><img src=""https://cdn-images-1.medium.com/max/1672/1*1lnQV5HUTpi97KwVgN1j_A.png"" width=""1672"" /></a></p><p class=""medium-feed-snippet"">You spend years learning the craft. Writing the pages. Recording the songs. Making something worth sharing.</p><p class=""medium-feed-link""><a href=""https://johnadamledford.medium.com/ai-wants-our-work-ask-us-first-b39c145661dc?source=rss------artificial_intelligence-5"">Continue reading on Medium »</a></p></div>"
4,To AI or Not to AI. The Slop Problem That Misses the Point.,NineTwoFive365,"Thu, 10 Sep 2026 12:31:01 GMT",https://medium.com/@ninetwofive365/to-ai-or-not-to-ai-the-slop-problem-that-misses-the-point-300d673521f6?source=rss-

profile for medium

In [ ]:
profile_data(df_medium, source_name= "medium")

--- Profiling Report: medium ---
Row Count: 268
Column Count: 6



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
title,object,0,0.0,229,CREATING CONSISTENT POSTS WITH QUOCIA
author,object,0,0.0,216,Candy Chan
published_date,object,0,0.0,218,"Thu, 10 Sep 2026 12:31:51 GMT"
url,object,0,0.0,268,https://medium.com/@candy_cyy/creating-consistent-posts-with-quocia-aa316f8a406f?source=rss------artificial_intelligence-5
tags,object,0,0.0,229,"[marketing, artificial-intelligence, quocia, ai, social-media]"
summary,object,0,0.0,268,"<div class=""medium-feed-item""><p class=""medium-feed-image""><a href=""https://medium.com/@candy_cyy/creating-consistent-posts-with-quocia-aa316f8a406f?source=rss------artificial_intelligence-5""><img src=""https://cdn-images-1.medium.com/max/1408/1*Mykxoyetl0mRaZF7Ovi1dg.jpeg"" width=""1408"" /></a></p><p class=""medium-feed-snippet"">Consistency is the single most decisive factor separating thriving digital brands from those that fade into irrelevance.</p><p class=""medium-feed-link""><a href=""https://medium.com/@candy_cyy/creating-consistent-posts-with-quocia-aa316f8a406f?source=rss------artificial_intelligence-5"">Continue reading on Medium »</a></p></div>"


checking duplicate titles


In [ ]:
# Count how many times each title appears
title_counts = df_medium['title'].value_counts()

# Filter to show only the titles that appear more than once
duplicated_titles = title_counts[title_counts > 1]

print("Titles with duplicates:")
display(duplicated_titles.head(10))

Titles with duplicates:


title
From Excel to Power BI: How I Built an Automated Data Pipeline Using Azure (Part 3)                  5
Your agent can read the whole internet. It cannot read flag_2.                                       4
Building an Interactive Data Quality Power BI Dashboard (Part 2)                                     4
Antusias dan Serunya Menggunakan Tencent EdgeOne Makers : Langkah Awal Membangun Project AI…         4
iGaming Compliance by Design: How to Build Infrastructure That Is Ready for MGA, PCI DSS and ISO…    3
The Modern Python Setup: Ditching Pip and Virtualenv for uv                                          3
The dbt CI Problem Nobody Talks About: Building Production-Faithful CI for Incremental Models        3
Your AI Agent Doesn’t Need to Be Malicious to Become a Security Problem                              2
Databricks SQL and Data Warehousing                                                                  2
From AI Pilot to Production: AI Consultants With Proven Deployment 

In [ ]:
pd.set_option("display.max_colwidth", None)
duplicated_titles = df_medium[df_medium["title"] == "From Excel to Power BI: How I Built an Automated Data Pipeline Using Azure (Part 3)"]
print(duplicated_titles["url"])

121      https://medium.com/@mandaricsladjana/from-excel-to-power-bi-how-i-built-an-automated-data-pipeline-using-azure-part-3-626762fefa75?source=rss------data_engineering-5
159                 https://medium.com/@mandaricsladjana/from-excel-to-power-bi-how-i-built-an-automated-data-pipeline-using-azure-part-3-626762fefa75?source=rss------azure-5
239         https://medium.com/@mandaricsladjana/from-excel-to-power-bi-how-i-built-an-automated-data-pipeline-using-azure-part-3-626762fefa75?source=rss------data_analysis-5
249    https://medium.com/@mandaricsladjana/from-excel-to-power-bi-how-i-built-an-automated-data-pipeline-using-azure-part-3-626762fefa75?source=rss------data_visualization-5
258               https://medium.com/@mandaricsladjana/from-excel-to-power-bi-how-i-built-an-automated-data-pipeline-using-azure-part-3-626762fefa75?source=rss------powerbi-5
Name: url, dtype: object


duplicated content needs to be dropped

In [ ]:
# 1. Sort by date so the newest version of the article is first
df_medium = df_medium.sort_values('published_date', ascending=False)

# 2. Drop duplicates looking ONLY at the title and author columns, keeping the first (newest) row
df_medium = df_medium.drop_duplicates(subset=['title', 'author'], keep='first')

# 3. Check how many rows are left
print(f"Rows remaining after removing title spam: {len(df_medium)}")

Rows remaining after removing title spam: 229


profile again

In [ ]:
profile_data(df_medium, source_name= "medium")

--- Profiling Report: medium ---
Row Count: 229
Column Count: 6



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
title,object,0,0.0,229,Telco AI at the Intersection of Network (OSS) and Business (BSS) Data — Part 2.
author,object,0,0.0,216,Ajay Nair
published_date,object,0,0.0,217,"Wed, 09 Sep 2026 23:30:18 GMT"
url,object,0,0.0,229,https://medium.com/@signalsandintel/telco-ai-at-the-intersection-of-network-oss-and-business-bss-data-part-2-cd2edaf30777?source=rss------data_architectures-5
tags,object,0,0.0,228,"[data-architecture, digital-transformation, telecommunication, artificial-intelligence, machine-learning]"
summary,object,0,0.0,229,"<div class=""medium-feed-item""><p class=""medium-feed-image""><a href=""https://medium.com/@signalsandintel/telco-ai-at-the-intersection-of-network-oss-and-business-bss-data-part-2-cd2edaf30777?source=rss------data_architectures-5""><img src=""https://cdn-images-1.medium.com/max/1408/1*aGU1bJS6Dl8KYVwhz8YB0Q.jpeg"" width=""1408"" /></a></p><p class=""medium-feed-snippet"">The story so far&#x2026;</p><p class=""medium-feed-link""><a href=""https://medium.com/@signalsandintel/telco-ai-at-the-intersection-of-network-oss-and-business-bss-data-part-2-cd2edaf30777?source=rss------data_architectures-5"">Continue reading on Medium »</a></p></div>"


cleaning the html from the summary in source medium

In [ ]:
from bs4 import BeautifulSoup

df_medium["summary_clean"] = df_medium["summary"].apply(
    lambda x: BeautifulSoup(x, "html.parser").get_text(" ", strip=True)
)
df_medium.head()

,title,author,published_date,url,tags,summary,summary_clean
133,Telco AI at the Intersection of Network (OSS) and Business (BSS) Data — Part 2.,Ajay Nair,"Wed, 09 Sep 2026 23:30:18 GMT",https://medium.com/@signalsandintel/telco-ai-at-the-intersection-of-network-oss-and-business-bss-data-part-2-cd2edaf30777?source=rss------data_architectures-5,"[data-architecture, digital-transformation, telecommunication, artificial-intelligence, machine-learning]","<div class=""medium-feed-item""><p class=""medium-feed-image""><a href=""https://medium.com/@signalsandintel/telco-ai-at-the-intersection-of-network-oss-and-business-bss-data-part-2-cd2edaf30777?source=rss------data_architectures-5""><img src=""https://cdn-images-1.medium.com/max/1408/1*aGU1bJS6Dl8KYVwhz8YB0Q.jpeg"" width=""1408"" /></a></p><p class=""medium-feed-snippet"">The story so far&#x2026;</p><p class=""medium-feed-link""><a href=""https://medium.com/@signalsandintel/telco-ai-at-the-intersection-of-network-oss-and-business-bss-data-part-2-cd2edaf30777?source=rss------data_architectures-5"">Continue reading on Medium »</a></p></div>",The story so far… Continue reading on Medium »
219,Modernizing Java Pet Store Without Turning It Into a Microservices Demo,Aditya Kulraj,"Wed, 09 Sep 2026 19:50:01 GMT",https://medium.com/@adityakulrajdtu/modernizing-java-pet-store-without-turning-it-into-a-microservices-demo-4a1ffbe2fbd6?source=rss------mongodb-5,"[mongodb, java, spring-boot, legacy-modernization, software-architecture]","<div class=""medium-feed-item""><p class=""medium-feed-snippet"">How I moved a J2EE 1.3-era application to Java 21 and Spring Boot while preserving its business behavior, supporting both Oracle and&#x2026;</p><p class=""medium-feed-link""><a href=""https://medium.com/@adityakulrajdtu/modernizing-java-pet-store-without-turning-it-into-a-microservices-demo-4a1ffbe2fbd6?source=rss------mongodb-5"">Continue reading on Medium »</a></p></div>","How I moved a J2EE 1.3-era application to Java 21 and Spring Boot while preserving its business behavior, supporting both Oracle and… Continue reading on Medium »"
192,Day 20 of DevOps: GKE Pod to Pod Communication,Shreya Sinha,"Wed, 09 Sep 2026 15:09:24 GMT",https://shreyajung.medium.com/day-20-of-devops-gke-pod-to-pod-communication-11af13e31ee3?source=rss------gcp-5,"[containers, gcp, kubernetes, docker, communication]","<div class=""medium-feed-item""><p class=""medium-feed-image""><a href=""https://shreyajung.medium.com/day-20-of-devops-gke-pod-to-pod-communication-11af13e31ee3?source=rss------gcp-5""><img src=""https://cdn-images-1.medium.com/max/2600/0*h5djBP8GksH5qSno"" width=""4427"" /></a></p><p class=""medium-feed-snippet"">Today we going to learn how pods in a kubernetes cluster communicate with each other.</p><p class=""medium-feed-link""><a href=""https://shreyajung.medium.com/day-20-of-devops-gke-pod-to-pod-communication-11af13e31ee3?source=rss------gcp-5"">Continue reading on Medium »</a></p></div>",Today we going to learn how pods in a kubernetes cluster communicate with each other. Continue reading on Medium »
182,**Africa is not becoming the largest continent.**,Oyediran Jomiloju Micheal,"Wed, 09 Sep 2026 14:18:44 GMT",https://medium.com/@oyediranjomiloju1/africa-is-not-becoming-the-largest-continent-adda4cc4d64c?source=rss------gis-5,"[environmental-impact, gis, africa, world]","<div class=""medium-feed-item""><p class=""medium-feed-image""><a href=""https://medium.com/@oyediranjomiloju1/africa-is-not-becoming-the-largest-continent-adda4cc4d64c?source=rss------gis-5""><img src=""https://cdn-images-1.medium.com/max/1536/1*5m0ve0c5xEIFtsnNXqH6nQ.png"" width=""1536"" /></a></p><p class=""medium-feed-snippet"">What is changing is not the size of Africa, but the way we represent its size on the world map. The map many of us grew up seeing does not&#x2026;</p><p class=""medium-feed-link""><a href=""https://medium.com/@oyediranjomiloju1/africa-is-not-becoming-the-largest-continent-adda4cc4d64c?source=rss------gis-

In [ ]:
#checking titles
df_medium["summary_clean"].head(20)

133                                                                                                                                        The story so far… Continue reading on Medium »
219                    How I moved a J2EE 1.3-era application to Java 21 and Spring Boot while preserving its business behavior, supporting both Oracle and… Continue reading on Medium »
192                                                                    Today we going to learn how pods in a kubernetes cluster communicate with each other. Continue reading on Medium »
182              What is changing is not the size of Africa, but the way we represent its size on the world map. The map many of us grew up seeing does not… Continue reading on Medium »
112                 If you’re new to a dbt project, there’s a chance your first real frustration will look something like this: you tweak one SQL file, run… Continue reading on Medium »
193                 Before you deploy applications to Google Cloud, yo

checking summary duplicates using profiling

In [ ]:
profile_data(df_medium, source_name= "medium")

--- Profiling Report: medium ---
Row Count: 229
Column Count: 7



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
title,object,0,0.0,229,Telco AI at the Intersection of Network (OSS) and Business (BSS) Data — Part 2.
author,object,0,0.0,216,Ajay Nair
published_date,object,0,0.0,217,"Wed, 09 Sep 2026 23:30:18 GMT"
url,object,0,0.0,229,https://medium.com/@signalsandintel/telco-ai-at-the-intersection-of-network-oss-and-business-bss-data-part-2-cd2edaf30777?source=rss------data_architectures-5
tags,object,0,0.0,228,"[data-architecture, digital-transformation, telecommunication, artificial-intelligence, machine-learning]"
summary,object,0,0.0,229,"<div class=""medium-feed-item""><p class=""medium-feed-image""><a href=""https://medium.com/@signalsandintel/telco-ai-at-the-intersection-of-network-oss-and-business-bss-data-part-2-cd2edaf30777?source=rss------data_architectures-5""><img src=""https://cdn-images-1.medium.com/max/1408/1*aGU1bJS6Dl8KYVwhz8YB0Q.jpeg"" width=""1408"" /></a></p><p class=""medium-feed-snippet"">The story so far&#x2026;</p><p class=""medium-feed-link""><a href=""https://medium.com/@signalsandintel/telco-ai-at-the-intersection-of-network-oss-and-business-bss-data-part-2-cd2edaf30777?source=rss------data_architectures-5"">Continue reading on Medium »</a></p></div>"
summary_clean,object,0,0.0,225,The story so far… Continue reading on Medium »


In [ ]:
# Count how many times each summary appears
summary_counts = df_medium['summary_clean'].value_counts()

# Filter to show only the titles that appear more than once
duplicated_summaries = summary_counts[summary_counts > 1]

print("summaries with duplicates:")
pd.set_option('display.max_colwidth', None)
duplicated_summaries.head(10)

summaries with duplicates:


summary_clean
Introduction Continue reading on Medium »    4
Continue reading on Medium »                 2
Name: count, dtype: int64

In [ ]:
dupe_summaries = df_medium[df_medium["summary_clean"] == "Introduction Continue reading on Medium »"]
print(dupe_summaries["url"])

156                      https://medium.com/@danielcarterus07/master-modern-workflows-using-data-integration-pentaho-solutions-4d0a1142c4f0?source=rss------data_integration-5
157                           https://aezion.medium.com/why-enterprise-integrations-become-harder-to-manage-as-businesses-grow-0b53c2808576?source=rss------data_integration-5
25     https://medium.com/@mail4haj/experienced-in-programming-yet-still-learning-reflections-from-my-first-week-of-a-data-science-f613a0bc78b0?source=rss------data_science-5
267      https://medium.com/@abinayabhi017/build-professional-business-dashboards-for-smarter-decision-making-power-bi-course-in-telugu-52f5a9700dbe?source=rss------powerbi-5
Name: url, dtype: object


load 5th source stackexchange

In [ ]:
stackexchange_file_path = "../data/raw/StackExchange_10-9-26_692_records.json"
df_stackexchange = pd.read_json(stackexchange_file_path)
df_stackexchange.head()

,question_id,title,author,published_date,url,tags,answer_count,score
0,80002178,IBKR ib_insync problem with placing stop limit orders in pre/post markets,SuperLeo,1789048958,https://stackoverflow.com/questions/80002178/ibkr-ib-insync-problem-with-placing-stop-limit-orders-in-pre-post-markets,"[python, ib-insync]",0,0
1,80002164,Is it okay to have a varied structure definition and implementation?,LakshyaK2011,1789046483,https://stackoverflow.com/questions/80002164/is-it-okay-to-have-a-varied-structure-definition-and-implementation,"[c, struct]",2,2
2,80002150,How can I test Windows-specific code for a cross-platform C++ project while developing on Linux?,mohamad hadi,1789041665,https://stackoverflow.com/questions/80002150/how-can-i-test-windows-specific-code-for-a-cross-platform-c-project-while-deve,"[advice, c++, linux, windows, cmake]",0,-1
3,80002173,Is it possible to make a component of a FlowLayout take up all available space?,Jurta,1789047943,https://stackoverflow.com/questions/80002173/is-it-possible-to-make-a-component-of-a-flowlayout-take-up-all-available-space,"[java, swing, jpanel, jtextfield, flowlayout]",0,-2
4,80002149,"Why is there no sorted_vector container in the C++ standard library, and what are the recommended alternatives?",Saint-Martin,1789041318,https://stackoverflow.com/questions/80002149/why-is-there-no-sorted-vector-container-in-the-c-standard-library-and-what-ar,"[best-practices, c++]",2,3


In [ ]:
df_stackexchange.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 692 entries, 0 to 691
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   question_id     692 non-null    int64 
 1   title           692 non-null    object
 2   author          692 non-null    object
 3   published_date  692 non-null    int64 
 4   url             692 non-null    object
 5   tags            692 non-null    object
 6   answer_count    692 non-null    int64 
 7   score           692 non-null    int64 
dtypes: int64(4), object(4)
memory usage: 43.4+ KB


profile of stackexchange code

In [ ]:
profile_data(df_stackexchange, source_name = "StackExchange")

--- Profiling Report: StackExchange ---
Row Count: 692
Column Count: 8



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
question_id,int64,0,0.0,692,80002178
title,object,0,0.0,692,IBKR ib_insync problem with placing stop limit orders in pre/post markets
author,object,0,0.0,678,SuperLeo
published_date,int64,0,0.0,692,1789048958
url,object,0,0.0,692,https://stackoverflow.com/questions/80002178/ibkr-ib-insync-problem-with-placing-stop-limit-orders-in-pre-post-markets
tags,object,0,0.0,666,"[python, ib-insync]"
answer_count,int64,0,0.0,20,0
score,int64,0,0.0,80,0


checking scores for the articles

In [ ]:
# Count how many times each score appears
score_counts = df_stackexchange['score'].value_counts()

# Filter to show only the scores that appear more than once
duplicated_scores = score_counts[score_counts > 1]

print("scores with duplicates:")
display(duplicated_scores.head)

scores with duplicates:


<bound method NDFrame.head of score
 0      158
 1      113
 2       80
 3       55
-1       38
 4       31
-2       25
 5       18
-4       14
-3       12
 7       11
 6       10
-5        9
-7        7
-6        6
 8        6
 18       5
 12       5
 11       4
 9        4
 10       4
 49       4
 24       3
 53       3
 32       3
-8        2
 19       2
 17       2
 120      2
 26       2
 29       2
 21       2
 39       2
 22       2
Name: count, dtype: int64>

In [ ]:
# Count how many times each answer_count appears
answer_count_counts = df_stackexchange['answer_count'].value_counts()

# Filter to show only the answer_counts that appear more than once
duplicated_answer_counts = answer_count_counts[answer_count_counts > 1]

print("answer_counts with duplicates:")
display(duplicated_answer_counts)

answer_counts with duplicates:


answer_count
1     256
2     160
0     101
3      67
4      41
5      16
6      15
7      12
9       7
8       4
11      2
10      2
17      2
Name: count, dtype: int64

fourth source Theguardian

In [ ]:
df_thegaurdian = pd.read_json("../data/raw/TheGuardian_10-9-26_2400_records.json")
df_thegaurdian.head()

,article_id,title,section,published_date,url,author,summary
0,technology/2026/sep/10/elon-musk-attacks-film-maker-behind-documentary-about-him,Elon Musk attacks film-maker behind documentary about him,Technology,2026-09-10T12:29:40Z,https://www.theguardian.com/technology/2026/sep/10/elon-musk-attacks-film-maker-behind-documentary-about-him,Michael Savage Media editor,Tech mogul calls director Alex Gibney a ‘bitter old man’ who has ‘zero integrity’ in series of posts on X
1,technology/2026/sep/10/openai-risk-catastrophic-loss-control-board-member-paul-christiano,"OpenAI not on track to reduce risk of ‘catastrophic’ loss of control, says board member",Technology,2026-09-10T12:23:21Z,https://www.theguardian.com/technology/2026/sep/10/openai-risk-catastrophic-loss-control-board-member-paul-christiano,Robert Booth UK technology editor,US government adviser Paul Christiano warns of risks to AI industry as he joins OpenAI’s non-profit foundation
2,technology/2026/sep/10/pixel-11-pro-review-googles-best-pocket-camera-goes-customisable,Pixel 11 Pro review: Google’s best pocket camera goes customisable,Technology,2026-09-10T06:00:19Z,https://www.theguardian.com/technology/2026/sep/10/pixel-11-pro-review-googles-best-pocket-camera-goes-customisable,Samuel Gibbs Consumer technology editor,"Solid battery life, snappy performance and quality software make for a great smaller phone with a killer camera"
3,technology/2026/sep/09/lawmakers-blast-ai-human-extinct-2030,Lawmakers blast AI companies after researcher warns of human extinction by 2030,Technology,2026-09-10T00:19:36Z,https://www.theguardian.com/technology/2026/sep/09/lawmakers-blast-ai-human-extinct-2030,Dara Kerr,Former Anthropic employee Jacob Coxon said AI will become ‘superhuman systems’ that can cause human extinction by the end of the decade
4,technology/2026/sep/09/anthropic-researchers-ai-human-extinction,Anthropic researchers say AI could cause human extinction by 2030,Technology,2026-09-10T00:14:08Z,https://www.theguardian.com/technology/2026/sep/09/anthropic-researchers-ai-human-extinction,Richard Luscombe,One researcher who resigned said many in industry believe that without regulation AI could kill off humanity


profile The gaurdian

In [ ]:
profile_data(df_thegaurdian, source_name = "The Gaurdian")

--- Profiling Report: The Gaurdian ---
Row Count: 2400
Column Count: 7



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
article_id,object,0,0.0,2400,technology/2026/sep/10/elon-musk-attacks-film-maker-behind-documentary-about-him
title,object,0,0.0,2400,Elon Musk attacks film-maker behind documentary about him
section,object,0,0.0,1,Technology
published_date,object,0,0.0,2383,2026-09-10T12:29:40Z
url,object,0,0.0,2400,https://www.theguardian.com/technology/2026/sep/10/elon-musk-attacks-film-maker-behind-documentary-about-him
author,object,120,5.0,653,Michael Savage Media editor
summary,object,0,0.0,2400,Tech mogul calls director Alex Gibney a ‘bitter old man’ who has ‘zero integrity’ in series of posts on X


source Towards Data Science

In [ ]:
towardsdatascience_file_path = "../data/raw/TowardsDataScience_10-9-26_20_records.json"
df_towardsdatascience = pd.read_json(towardsdatascience_file_path)
df_towardsdatascience.head()

,title,author,published_date,url,tags,summary
0,Optimizing LLM Inference Costs in Multi-Agent Systems with Adaptive Model Routing,Partha Sarkar,"Thu, 10 Sep 2026 12:30:01 GMT",https://towardsdatascience.com/optimizing-llm-inference-costs-in-multi-agent-systems-with-adaptive-model-routing/,"[Agentic AI, Agent Architecture, LLM Agents, Multi Agent Systems, LLM Inference, Inference]","<p>Moving from static model assignment to intelligent, task-level LLM selection.</p>\n<p>The post <a href=""https://towardsdatascience.com/optimizing-llm-inference-costs-in-multi-agent-systems-with-adaptive-model-routing/"">Optimizing LLM Inference Costs in Multi-Agent Systems with Adaptive Model Routing</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>"
1,Who Questions What Works: When Should We Retest Our Assumptions?,Erika Gomes-Gonçalves,"Thu, 10 Sep 2026 11:00:01 GMT",https://towardsdatascience.com/who-questions-what-works-when-should-we-retest-our-assumptions/,"[Data Science, Concept Drift, Ai Governance, Mlops, Risk Management, Editor's Picks]","<p>A model is only as reliable as the assumptions behind it</p>\n<p>The post <a href=""https://towardsdatascience.com/who-questions-what-works-when-should-we-retest-our-assumptions/"">Who Questions What Works: When Should We Retest Our Assumptions?</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>"
2,Getting started with dbt,Thomas Reid,"Wed, 09 Sep 2026 15:30:01 GMT",https://towardsdatascience.com/getting-started-with-dbt/,"[Data Engineering, Python, Dbt, Sql Analytics, Sql, Programming]","<p>A practical guide to building, testing, and documenting SQL transformations</p>\n<p>The post <a href=""https://towardsdatascience.com/getting-started-with-dbt/"">Getting started with dbt</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>"
3,The Symmetry That Breaks Neural Network Averaging,Ananya Bhattacharyya,"Wed, 09 Sep 2026 14:00:02 GMT",https://towardsdatascience.com/the-symmetry-that-breaks-neural-network-averaging/,"[Deep Learning, Neural Networks, Optimisation, Neural Network Training, Machine Learning, Editor's Picks]","<p>Understanding permutation symmetry in deep learning, and what it means for weight averaging and model merging</p>\n<p>The post <a href=""https://towardsdatascience.com/the-symmetry-that-breaks-neural-network-averaging/"">The Symmetry That Breaks Neural Network Averaging</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>"
4,When One Process Becomes Too Much: Splitting a Pipeline into MCP Services,Priyansh Bhardwaj,"Wed, 09 Sep 2026 12:30:02 GMT",https://towardsdatascience.com/when-one-process-becomes-too-much-splitting-a-pipeline-into-mcp-services/,"[Agentic AI, Agentic Workflows, Model Context Protocol, Llm Applications, Generative Ai, Editor's Picks]","<p>How we split a tightly coupled Python pipeline into independently deployable services</p>\n<p>The post <a href=""https://towardsdatascience.com/when-one-process-becomes-too-much-splitting-a-pipeline-into-mcp-services/"">When One Process Becomes Too Much: Splitting a Pipeline into MCP Services</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>"


cleaning the summary data

In [ ]:

# Clean the HTML from the summary column
def clean_summary(html):
    # Make sure the value is a string before processing
    if not isinstance(html, str):
        return ""

    # Parse the HTML
    soup = BeautifulSoup(html, "html.parser")

    # Get the first paragraph, which contains the actual article summary
    first_p = soup.find("p")

    # Return the text without HTML tags
    if first_p:
        return first_p.get_text(" ", strip=True)

    # Return an empty string if no paragraph was found
    return ""


# Apply the cleaning function to the summary column
df_towardsdatascience["clean_summary"] = df_towardsdatascience["summary"].apply(clean_summary)

In [ ]:
df_towardsdatascience.head()

,title,author,published_date,url,tags,summary,clean_summary
0,Optimizing LLM Inference Costs in Multi-Agent Systems with Adaptive Model Routing,Partha Sarkar,"Thu, 10 Sep 2026 12:30:01 GMT",https://towardsdatascience.com/optimizing-llm-inference-costs-in-multi-agent-systems-with-adaptive-model-routing/,"[Agentic AI, Agent Architecture, LLM Agents, Multi Agent Systems, LLM Inference, Inference]","<p>Moving from static model assignment to intelligent, task-level LLM selection.</p>\n<p>The post <a href=""https://towardsdatascience.com/optimizing-llm-inference-costs-in-multi-agent-systems-with-adaptive-model-routing/"">Optimizing LLM Inference Costs in Multi-Agent Systems with Adaptive Model Routing</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>","Moving from static model assignment to intelligent, task-level LLM selection."
1,Who Questions What Works: When Should We Retest Our Assumptions?,Erika Gomes-Gonçalves,"Thu, 10 Sep 2026 11:00:01 GMT",https://towardsdatascience.com/who-questions-what-works-when-should-we-retest-our-assumptions/,"[Data Science, Concept Drift, Ai Governance, Mlops, Risk Management, Editor's Picks]","<p>A model is only as reliable as the assumptions behind it</p>\n<p>The post <a href=""https://towardsdatascience.com/who-questions-what-works-when-should-we-retest-our-assumptions/"">Who Questions What Works: When Should We Retest Our Assumptions?</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>",A model is only as reliable as the assumptions behind it
2,Getting started with dbt,Thomas Reid,"Wed, 09 Sep 2026 15:30:01 GMT",https://towardsdatascience.com/getting-started-with-dbt/,"[Data Engineering, Python, Dbt, Sql Analytics, Sql, Programming]","<p>A practical guide to building, testing, and documenting SQL transformations</p>\n<p>The post <a href=""https://towardsdatascience.com/getting-started-with-dbt/"">Getting started with dbt</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>","A practical guide to building, testing, and documenting SQL transformations"
3,The Symmetry That Breaks Neural Network Averaging,Ananya Bhattacharyya,"Wed, 09 Sep 2026 14:00:02 GMT",https://towardsdatascience.com/the-symmetry-that-breaks-neural-network-averaging/,"[Deep Learning, Neural Networks, Optimisation, Neural Network Training, Machine Learning, Editor's Picks]","<p>Understanding permutation symmetry in deep learning, and what it means for weight averaging and model merging</p>\n<p>The post <a href=""https://towardsdatascience.com/the-symmetry-that-breaks-neural-network-averaging/"">The Symmetry That Breaks Neural Network Averaging</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>","Understanding permutation symmetry in deep learning, and what it means for weight averaging and model merging"
4,When One Process Becomes Too Much: Splitting a Pipeline into MCP Services,Priyansh Bhardwaj,"Wed, 09 Sep 2026 12:30:02 GMT",https://towardsdatascience.com/when-one-process-becomes-too-much-splitting-a-pipeline-into-mcp-services/,"[Agentic AI, Agentic Workflows, Model Context Protocol, Llm Applications, Generative Ai, Editor's Picks]","<p>How we split a tightly coupled Python pipeline into independently deployable services</p>\n<p>The post <a href=""https://towardsdatascience.com/when-one-process-becomes-too-much-splitting-a-pipeline-into-mcp-services/"">When One Process Becomes Too Much: Splitting a Pipeline into MCP Services</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>",How we split a tightly coupled Python pipeline into independently deployable services


profile again

In [ ]:
profile_data(df_towardsdatascience, source_name = "Towards Data Science")


--- Profiling Report: Towards Data Science ---
Row Count: 20
Column Count: 7



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
title,object,0,0.0,20,Optimizing LLM Inference Costs in Multi-Agent Systems with Adaptive Model Routing
author,object,0,0.0,19,Partha Sarkar
published_date,object,0,0.0,20,"Thu, 10 Sep 2026 12:30:01 GMT"
url,object,0,0.0,20,https://towardsdatascience.com/optimizing-llm-inference-costs-in-multi-agent-systems-with-adaptive-model-routing/
tags,object,0,0.0,20,"[Agentic AI, Agent Architecture, LLM Agents, Multi Agent Systems, LLM Inference, Inference]"
summary,object,0,0.0,20,"<p>Moving from static model assignment to intelligent, task-level LLM selection.</p>\n<p>The post <a href=""https://towardsdatascience.com/optimizing-llm-inference-costs-in-multi-agent-systems-with-adaptive-model-routing/"">Optimizing LLM Inference Costs in Multi-Agent Systems with Adaptive Model Routing</a> appeared first on <a href=""https://towardsdatascience.com"">Towards Data Science</a>.</p>"
clean_summary,object,0,0.0,20,"Moving from static model assignment to intelligent, task-level LLM selection."


Hacker news

In [ ]:
import glob

# Get all JSON files from the HackerNews folder
files = glob.glob("../data/raw/HackerNews/*.json")

# Read each JSON file into a DataFrame
dfs = [pd.read_json(file) for file in files]

print(dfs)
# Combine all DataFrames into one
df_hackernews = pd.concat(dfs, ignore_index=True)

[     article_id  \
0       6393724   
1      40397637   
2      18459016   
3      12786545   
4      42790207   
..          ...   
945    27456831   
946     2740827   
947     2647293   
948    23239118   
949     2302975   

                                                                               title  \
0                 Cypersecurity pro on Nasdaq website: 'I needed 10 minutes to hack'   
1                                            Cyber Security: A pre-war reality check   
2                        Japan's cyber-security minister has 'never used a computer'   
3                                                     Introduction to Cyber Security   
4    DHS removes all members of cyber security advisory boards, halts investigations   
..                                                                               ...   
945                                                                     Scholarships   
946                                       Ask HN: Openly posting s

In [ ]:
df_hackernews.head()

,article_id,title,author,published_date,url,points,num_comments
0,6393724,Cypersecurity pro on Nasdaq website: 'I needed 10 minutes to hack',jalanco,2013-09-16T15:30:29Z,http://www.nydailynews.com/news/crime/cypersecurity-pro-hacks-nasdaq-10-minutes-article-1.1456844,1,0
1,40397637,Cyber Security: A pre-war reality check,edent,2024-05-18T09:38:08Z,https://berthub.eu/articles/posts/cyber-security-pre-war-reality-check/,593,278
2,18459016,Japan's cyber-security minister has 'never used a computer',GeneticGenesis,2018-11-15T13:21:49Z,https://www.bbc.com/news/technology-46222026,481,318
3,12786545,Introduction to Cyber Security,nygrenh,2016-10-25T09:44:37Z,https://cybersecuritybase.github.io/,305,40
4,42790207,"DHS removes all members of cyber security advisory boards, halts investigations",BHSPitMonkey,2025-01-22T07:53:28Z,https://bsky.app/profile/ericjgeller.com/post/3lgbpqmxeok2f,300,368


profile of hackernews

In [ ]:
profile_data(df_hackernews)

--- Profiling Report: Dataset ---
Row Count: 4750
Column Count: 7



,Data Type,Null Count,Percent Null,Unique Count,Sample Value
article_id,int64,0,0.0,4734,6393724
title,object,0,0.0,4485,Cypersecurity pro on Nasdaq website: 'I needed 10 minutes to hack'
author,object,0,0.0,3494,jalanco
published_date,object,0,0.0,4734,2013-09-16T15:30:29Z
url,object,684,14.4,3713,http://www.nydailynews.com/news/crime/cypersecurity-pro-hacks-nasdaq-10-minutes-article-1.1456844
points,int64,0,0.0,736,1
num_comments,int64,0,0.0,628,0


remove duplicate urls

In [ ]:
# 1. Sort by date so the newest version of the article is first
df_hackernews = df_hackernews.sort_values('published_date', ascending=False)

# 2. Drop duplicates looking ONLY at the url columns, keeping the first (newest) row
df_hackernews = df_hackernews.drop_duplicates(subset=["url"], keep='first')

# 3. Check how many rows are left
print(f"Rows remaining after removing url dupes: {len(df_hackernews)}")

Rows remaining after removing url dupes: 3713


seeing duplicated titles

In [ ]:
# Count how many times each title appears
title_counts = df_hackernews['title'].value_counts()
# Filter to show only the titles that appear more than once
duplicated_summaries = title_counts[title_counts > 1]

print("title with duplicates:")
pd.set_option('display.max_colwidth', None)
duplicated_summaries.head(10)

summaries with duplicates:


title
Foundations of Data Science [pdf]                                         5
Data Wrangling at Slack                                                   3
10 Modern Software Over-Engineering Mistakes                              3
No Silver Bullet: Essence and Accidents of Software Engineering (1987)    3
Frequently Forgotten Fundamental Facts about Software Engineering         3
Software Engineering at Google                                            3
Hackers Courted by Government for Cyber Security Jobs                     3
Data Engineering with Rust                                                3
SAP Cyber Security in Figures: Global Threat Report 2016                  3
What Is Data Engineering?                                                 3
Name: count, dtype: int64

check the title duplicates

In [ ]:
duped_titles = df_hackernews[df_hackernews["title"] == "Hackers Courted by Government for Cyber Security Jobs"]
duped_titles["url"]

534    http://www.rollingstone.com/feature/the-geeks-on-the-frontlines#i.1qnflh7uzmdgpz
278    http://www.rollingstone.com/feature/the-geeks-on-the-frontlines#i.1m2l24r9edjws1
82     http://www.rollingstone.com/feature/the-geeks-on-the-frontlines#i.ynmf8114dxfiky
Name: url, dtype: object

# **Mouath's PART**

## 1. Ingestion: Load Partitioned Hacker News JSONs

In [ ]:
import pandas as pd
import json
from google.colab import files

print("Select the HackerNews JSON files from your local Downloads folder...")
# This will open a file picker dialog in your browser
uploaded = files.upload()

def process_uploaded_jsons(uploaded_dict: dict) -> pd.DataFrame:
    all_records = []

    for filename, content in uploaded_dict.items():
        try:
            # Colab uploads files as raw bytes, so we must decode to UTF-8 first
            data = json.loads(content.decode('utf-8'))

            if isinstance(data, list):
                all_records.extend(data)
                print(f"✅ Loaded {len(data)} records from {filename}")
            else:
                print(f"⚠️ Warning: {filename} did not contain a JSON list. Skipping.")

        except Exception as e:
            print(f"❌ Failed to parse {filename}: {e}")

    return pd.DataFrame(all_records)

df_raw = process_uploaded_jsons(uploaded)
print(f"\nUnified Raw DataFrame loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns.")

Select the HackerNews JSON files from your local Downloads folder...


Saving HackerNews-DataEng_10-9-26_950_records.json to HackerNews-DataEng_10-9-26_950_records.json
Saving HackerNews-SoftwareEng_10-9-26_950_records.json to HackerNews-SoftwareEng_10-9-26_950_records.json
Saving HackerNews-DataScience_10-9-26_950_records.json to HackerNews-DataScience_10-9-26_950_records.json
Saving HackerNews-CyperSec_10-9-26_950_records.json to HackerNews-CyperSec_10-9-26_950_records.json
Saving HackerNews-AI_10-9-26_950_records.json to HackerNews-AI_10-9-26_950_records.json
✅ Loaded 950 records from HackerNews-DataEng_10-9-26_950_records.json
✅ Loaded 950 records from HackerNews-SoftwareEng_10-9-26_950_records.json
✅ Loaded 950 records from HackerNews-DataScience_10-9-26_950_records.json
✅ Loaded 950 records from HackerNews-CyperSec_10-9-26_950_records.json
✅ Loaded 950 records from HackerNews-AI_10-9-26_950_records.json

Unified Raw DataFrame loaded: 4,750 rows, 7 columns.


In [ ]:
import pandas as pd
from urllib.parse import urlparse, urlunparse

def clean_hackernews(df: pd.DataFrame) -> pd.DataFrame:
    clean_df = df.copy()

    # 1. Standardize Timestamps
    # Converts ISO strings (e.g., "2013-09-16T15:30:29Z") to Pandas UTC datetime
    clean_df['published_date'] = pd.to_datetime(clean_df['published_date'], errors='coerce', utc=True)

    # 2. Impute Null URLs for "Ask HN" / "Tell HN" posts
    # If URL is null, construct the native YCombinator thread URL using the article_id
    clean_df['url'] = clean_df.apply(
        lambda row: f"https://news.ycombinator.com/item?id={row['article_id']}"
        if pd.isna(row['url']) or str(row['url']).strip() == ""
        else row['url'],
        axis=1
    )

    # 4. Chronological Deduplication
    # Sort by date descending
    clean_df = clean_df.sort_values('published_date', ascending=False)

    # Drop duplicates based on the cleaned URL, keeping the newest
    initial_count = len(clean_df)
    clean_df = clean_df.drop_duplicates(subset=['url'], keep='first')

    print(f"URL Deduplication: Dropped {initial_count - len(clean_df)} duplicate records.")

    # 5. Type Casting
    clean_df['points'] = pd.to_numeric(clean_df['points'], errors='coerce').fillna(0).astype(int)
    clean_df['num_comments'] = pd.to_numeric(clean_df['num_comments'], errors='coerce').fillna(0).astype(int)

    return clean_df

# Execute cleaning
df_hn_clean = clean_hackernews(df_raw)

# Profile the cleaned dataset using your colleague's function concept
print(f"\nFinal Cleaned Shape: {df_hn_clean.shape[0]} rows, {df_hn_clean.shape[1]} columns")

URL Deduplication: Dropped 1116 duplicate records.

Final Cleaned Shape: 3634 rows, 8 columns


In [ ]:
df_hn_clean.head()

,article_id,title,author,published_date,url,points,num_comments,canonical_url
3138,49633428,Ex-Microsoft cyber security SME shares insight...,Falconeer,2026-09-09 20:08:14+00:00,https://www.activedirectorysecurity.com,2,0,https://www.activedirectorysecurity.com
4381,49630253,AirPods 5,awad,2026-09-09 17:39:24+00:00,https://www.apple.com/newsroom/2026/09/apple-i...,484,411,https://www.apple.com/newsroom/2026/09/apple-i...
4402,49616968,Tao: Open math problems being non-renewably mi...,_alternator_,2026-09-08 21:00:52+00:00,https://mathstodon.xyz/@tao/117237320796901560,476,411,https://mathstodon.xyz/@tao/117237320796901560
4090,49615537,Muse – Meta’s personal AI agent,yks,2026-09-08 19:25:00+00:00,https://ai.meta.com/muse/,646,724,https://ai.meta.com/muse/
1364,49610908,I think Astra is absolutely amazing but I'm ba...,tosh,2026-09-08 14:34:38+00:00,https://twitter.com/mitsuhiko/status/209731825...,11,2,https://twitter.com/mitsuhiko/status/209731825...


In [ ]:
df_hn_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3634 entries, 3138 to 1439
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   article_id      3634 non-null   object             
 1   title           3634 non-null   object             
 2   author          3634 non-null   object             
 3   published_date  3634 non-null   datetime64[ns, UTC]
 4   url             3634 non-null   object             
 5   points          3634 non-null   int64              
 6   num_comments    3634 non-null   int64              
 7   canonical_url   3634 non-null   object             
dtypes: datetime64[ns, UTC](1), int64(2), object(5)
memory usage: 255.5+ KB


In [ ]:
df_hn_clean['article_id'] = pd.to_numeric(df_hn_clean['article_id'], errors='coerce').astype('Int64')
df_hn_clean['published_date'] = pd.to_datetime(df_hn_clean['published_date'])

In [ ]:
df_hn_clean.to_csv('hackernews_cleaned.csv', index=False)

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

# 1. Profiling Function
def profile_data(df, source_name="Dataset"):
    print(f"Profiling Report: {source_name}")
    print(f"Row Count: {df.shape[0]}")
    print(f"Column Count: {df.shape[1]}\n")

    # Safely count uniques by treating lists/dicts as strings
    safe_unique_counts = [df[col].astype(str).nunique() for col in df.columns]

    profiling_summary = pd.DataFrame({
        'Data Type': df.dtypes,
        'Null Count': df.isnull().sum(),
        'Percent Null': (df.isnull().sum() / len(df)) * 100,
        'Unique Count': safe_unique_counts,
        'Sample Value': df.iloc[0] if not df.empty else None
    })

    return profiling_summary

# Execute Initial Profile
print(profile_data(df_raw, source_name="HackerNews (Raw)"))

# 2. Chronological Deduplication
# Sort by date so the newest version of the article is first
df_hackernews = df_raw.sort_values('published_date', ascending=False)

# Drop duplicates looking ONLY at the url column, keeping the first (newest) row
df_hackernews = df_hackernews.drop_duplicates(subset=["url"], keep='first')
print(f"\nRows remaining after removing url dupes: {len(df_hackernews)}")

# 3. Duplicate Title Auditing
# Count how many times each title appears
title_counts = df_hackernews['title'].value_counts()
duplicated_titles = title_counts[title_counts > 1]
print("\nTitles with duplicates:")
print(duplicated_titles.head(10))

# 4. HTML Sanitization (Conditionally applied if the source has a summary field)
def clean_summary(html):
    if pd.isna(html) or not isinstance(html, str):
        return ""
    soup = BeautifulSoup(html, "html.parser")
    first_p = soup.find("p")
    if first_p:
        return first_p.get_text(" ", strip=True)
    return ""

if 'summary' in df_hackernews.columns:
    df_hackernews["clean_summary"] = df_hackernews["summary"].apply(clean_summary)

# Execute Final Profile
print("\n")
print(profile_data(df_hackernews, source_name="HackerNews (Cleaned)"))

Profiling Report: HackerNews (Raw)
Row Count: 4750
Column Count: 7

               Data Type  Null Count  Percent Null  Unique Count  \
article_id        object           0           0.0          4734   
title             object           0           0.0          4485   
author            object           0           0.0          3494   
published_date    object           0           0.0          4734   
url               object         684          14.4          3713   
points             int64           0           0.0           736   
num_comments       int64           0           0.0           628   

                                       Sample Value  
article_id                                 20554194  
title                     Data Engineering Cookbook  
author                                     charlysl  
published_date                 2019-07-29T13:08:26Z  
url             https://github.com/andkret/Cookbook  
points                                          382  
num_comme

In [ ]:
print(profile_data(df_hn_clean, source_name= "HackerNews after Cleaning"))

Profiling Report: HackerNews after Cleaning
Row Count: 3634
Column Count: 7

                          Data Type  Null Count  Percent Null  Unique Count  \
article_id                    Int64           0           0.0          3634   
title                        object           0           0.0          3580   
author                       object           0           0.0          2700   
published_date  datetime64[ns, UTC]           0           0.0          3634   
url                          object           0           0.0          3634   
points                        int64           0           0.0           708   
num_comments                  int64           0           0.0           606   

                                                     Sample Value  
article_id                                               49633428  
title           Ex-Microsoft cyber security SME shares insight...  
author                                                  Falconeer  
published_date    

In [2]:
!pip install GitPython
import git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/222.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 5.3 MB/s eta 0:00:00
